# 🚀 Stack 2.9 - Kaggle Training Notebook

**Free GPU training on Kaggle**

This notebook trains a LoRA adapter for Stack 2.9 on **Qwen2.5-Coder-7B** using Kaggle's free GPU.

⏱️ **Expected runtime:** 2-4 hours
💾 **VRAM needed:** ~16GB (Kaggle P100 has 16GB)

---

**Instructions:**
1. Kaggle → New Notebook
2. Add this notebook's code OR clone from GitHub
3. Enable GPU (Settings → Accelerator → GPU P100)
4. Run cells in order

---

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# STEP 1: Clone the repo
import os
import shutil

REPO_DIR = "/kaggle/working/stack-2.9"

# Remove old if exists
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/my-ai-stack/stack-2.9.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"✅ Working in: {os.getcwd()}")

In [ ]:
# STEP 2: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers peft accelerate datasets pyyaml tqdm scipy bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# STEP 3: Download Base Model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B"
MODEL_DIR = os.path.join(REPO_DIR, "base_model_qwen7b")

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print(f"Downloading {MODEL_NAME}...")
    print("This takes ~10-15 minutes...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.save_pretrained(MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model.save_pretrained(MODEL_DIR)
    print("✅ Model downloaded!")
else:
    print("✅ Model already exists")

!ls -lh {MODEL_DIR} | head -5

In [ ]:
# STEP 4: Setup paths and config
import yaml

config_path = os.path.join(REPO_DIR, "stack/training/train_config_local.yaml")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update for Kaggle GPU
config['model']['name'] = MODEL_DIR
config['hardware']['device'] = "cuda"
config['hardware']['num_gpus'] = 1

OUTPUT_DIR = os.path.join(REPO_DIR, "training_output")
config['output']['lora_dir'] = os.path.join(OUTPUT_DIR, "lora")
config['output']['merged_dir'] = os.path.join(OUTPUT_DIR, "merged")

os.makedirs(OUTPUT_DIR, exist_ok=True)
updated_config = os.path.join(OUTPUT_DIR, "train_config.yaml")

with open(updated_config, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config saved to: {updated_config}")
print(f"   Device: {config['hardware']['device']}")

In [ ]:
# STEP 5: Train LoRA
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))

print("="*60)
print("STARTING TRAINING")
print("="*60)

from train_lora import train_lora
trainer = train_lora(updated_config)

print("="*60)
print("TRAINING COMPLETED")
print("="*60)

In [ ]:
# STEP 6: Merge and save
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))
from merge_adapter import merge_adapter

merged_dir = os.path.join(OUTPUT_DIR, "merged")
os.makedirs(merged_dir, exist_ok=True)

merge_config = {
    'model': {'name': MODEL_DIR, 'trust_remote_code': True},
    'output': {'lora_dir': os.path.join(OUTPUT_DIR, 'lora'), 'merged_dir': merged_dir},
    'quantization': {'enabled': False}
}

merge_cfg_path = os.path.join(OUTPUT_DIR, "merge_config.yaml")
with open(merge_cfg_path, 'w') as f:
    yaml.dump(merge_config, f)

merge_adapter(merge_cfg_path, os.path.join(OUTPUT_DIR, "lora"), merged_dir)

print(f"✅ Model saved to: {merged_dir}")
!ls -lh {merged_dir}

In [ ]:
# STEP 7: Download the trained model (for saving)
# The model is saved at OUTPUT_DIR/merged/
# You can download it from the Kaggle outputs

print("Training complete!")
print(f"Model saved at: {merged_dir}")
print("\nTo download:")
print("1. Click 'Output' tab in Kaggle")
print("2. Download the files from training_output/merged/")